# មេរៀន ១៨៖ ការសាងសង់សុវត្ថិភាពអ្នកភ្នាក់ងារ AI ជាមួយបReceipt ប្រើបច្ចេកវិទ្យាអ៊ិនគ្រីប

## សៀវភៅកំណត់ត្រាដៃគូ

សៀវភៅកំណត់ត្រានេះនាំយើងរត់តាមលំហាត់បួន៖

១. **ចុះហត្ថលេខាលើបReceipt ដំបូងរបស់អ្នក** សម្រាប់ការហៅឧបករណ៍អ្នកភ្នាក់ងារ ហើយធ្វើការផ្ទៀងផ្ទាត់វា។
២. **បំភ្លាញ់បReceipt** ហើយមើលការផ្ទៀងផ្ទាត់បរាជ័យ។
៣. **កសាងខ្សែបReceipt បី** ហើយបញ្ជាក់ភាពខ្ចាប់ខ្ចាយនៃខ្សែ។
៤. **បញ្ចប់ការហៅឧបករណ៍ Microsoft Agent Framework** ដូច្នេះកិច្ចការត្រូវបញ្ចេញបReceipt រាល់ដំណើរការ។

ពីរដើមគ្រឿងអេលិកត្រូនិចគ្រប់ៗគ្នាត្រូវបាននាំចូលពីបណ្ណាល័យដែលថែទាំយ៉ាងល្អ (`pynacl` សម្រាប់ Ed25519, `jcs` សម្រាប់ RFC 8785 canonical JSON, `hashlib` ពីបណ្ណាល័យស្តង់ដារ Python សម្រាប់ SHA-256)។ តម្រឹមលទ្ធភាពបReceipt គឺជាភាសា Python ដើមដែលអ្នកអាចអាន និងកែប្រែបាន។

រត់កូដទាំងអស់តាមលំដាប់។ ផ្នែកនីមួយៗមានអត្ថន័យខ្លី និងឯករាជ្យ។


## ការតំឡើង

ตំឡើងផ្នែកខាងគាំទ្រទាំងពីរ។ ទាំងពីរមានបណ្ណាការអនុញ្ញាតបើកចំហ (Apache-2.0 / MIT)។


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## ឧបករណ៍ជំនួយ

ឧបករណ៍ជំនួយទាំងពីរនេះដោះស្រាយការអ៊ិនកូដ base64url (គ្មានការដាក់បន្ថែម) និងការឯកទេស SHA-256 នៃវត្ថុទូទៅណាមួយ។ ពួកវាអោយចំណាប់អារម្មណ៍នៅចំណុចសំខាន់ស្តីពីល្បឿននៃបង្កាន់ដៃ។


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## ផ្នែក 1៖ ចុះហត្ថអក្សរលើបង្កាន់ដៃដំបូងរបស់អ្នក

គិតទៅអេជនបុគ្គលិករបស់យើងសម្រាប់ **Contoso Travel** ដែលទើបតែស្វែងរករូបិយវត្ថុខេត្តពី Sydney ទៅ Los Angeles សម្រាប់អតិថិជនម្នាក់។ យើងចង់កត់ត្រាសេចក្តីថ្លែងការណ៍របស់កាតព្វកិច្ចនេះអំពីការហៅឧបករណ៍នេះដើម្បីឲ្យអ្នកត្រួតពិនិត្យនៅពេលអនាគតអាចបញ្ជាក់វាទៅក្រោមកូនសោអ្នកចេញផ្សាយដែលបានចងភ្ជាប់បំបែក។

### ជំហាន 1.1៖ បង្កើតកូនសោសម្រាប់ចុះហត្ថអក្សរ

នៅក្នុងការផលិត កូនសោសម្រាប់ចុះហត្ថអក្សររបស់អេជននឹងរស់នៅក្នុងឧបករណ៍សន្តិសុខរឹង (HSM), Azure Key Vault ឬឃ្លាំងដែលបានការពារដូចគ្នា។ សម្រាប់មេរៀននេះ យើងបង្កើតកូនសោថ្មីនៅក្នុងអង្គចងចាំ។


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### ជំហាន ១.២៖ សង់ផ្ទៃសារព័ត៌មានបង្កាន់ដៃ

ផ្ទៃសារព័ត៌មានមានអ្វីគ្រប់យ៉ាងដែលយើងចង់ឲ្យបង្កាន់ដៃបញ្ជាក់៖ នរណាទៅដើម, ដំណើរការអ្វី, ការប្រើឧបករណ៍អ្វី, ជាមួយអ្វីនៃអ្វី, តើអ្វីបានត្រឡប់មកវិញ, ក្រោមគោលការណ៍អ្វី, និងពេលណា។ យើងធ្វើការ hash ប៉ារ៉ាម៉ែត្រ និងលទ្ធផលជំនួសការបញ្ចូលពួកវាទៅក្នុងបន្ទាត់ ដូច្នេះបង្កាន់ដៃមិនដូរសារសុវត្ថិភាព។ 


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### ជំហាន 1.3: ចុះហត្ថលេខា និងប្រមូលផ្តុំវិកយប័ត្រ

បីជំហាន៖

1. បម្លែងទម្រង់ payload ដោយប្រើ JCS ដើម្បីឲ្យកម្មវិធីពីរដែលផលិតវិក័យប័ត្រដូចគ្នាអាចផលិតប៊ៃតស្មើគ្នាតាមអក្ខរក្រម។
2. ចុះហត្ថលេខាលើប៊ៃតទម្រង់ canonical ដោយផ្ទាល់ជាមួយកូនសោឯកជន Ed25519 ។ PureEdDSA រៀបចំសារជា hash នៅក្នុងខ្លួនវា ដូច្នេះការបន្ថែមផលលទ្ធផល pre-hash ផ្សេងទៀតនឹងផ្លាស់ប្ដូរពិធីសាស្រ្ត។

ហត្ថលេខានឹងត្រូវភ្ជាប់ជាមួយ payload ដើមឲ្យបានវិក័យប័ត្រជាចុងក្រោយ។


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### ជំហាន 1.4: ពិនិត្យបង្កប់ការទទួល

ការពិនិត្យបញ្ជូនត្រឡប់ដំណើរការ ។ យើងដកហ្ស៊ីណាតូរ, គណនាមេតិការ canonical ផងឡើងវិញ, ហើយពិនិត្យហ្ស៊ីណាតូរ ប្រៀបធៀបទៅនឹងកូនសោសាធារណៈរបស់អ្នកចេញផ្សាយ។

ការពិនិត្យបញ្ជូនធ្វើនៅក្រៅបណ្តាញ: គ្មានសេវាកម្មត្រូវហៅ និងគ្មានការចូលប្រើបណ្តាញ។ វាត្រូវការអ្វីមួយ មិនមែនត្រឹមតែបង្កប់ទទួលទេ: គឺមានកូនសោសាធារណៈរបស់អ្នកចេញផ្សាយ ដែលទទួលបានតាមបណ្តាញ ដែលអ្នកពិនិត្យឯកសារបានជឿជាក់រួចហើយ, ដូចជា JWK Set ដែលអ្នកចេញផ្សាយផ្សព្វផ្សាយ ឬតម្លៃដែលបានដាក់ជាមុន។ កន្លែង `public_key` ក្នុងបង្កប់ទទួលមិនអាចចូលរួមតួនាទីនោះបាន​ទេ ពីព្រោះអ្នកដែលសរសេរបង្កប់ទទួលនោះក៏សរសេរកន្លែងនោះដែរ។ មេរៀនទី ២ បង្ហាញពីការបោកប្រាស់ដែលការពារនេះបានរាំងស្ទះ។


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


អ្នកគួរតែឃើញ `Receipt is valid: True`, `Pre-hashed receipt valid: False` និង `Malformed receipts rejected: True`។ ករណីវិជ្ជមានបញ្ជាក់ថាចំណត់លេខរាង JCS ដោយផ្ទាល់ដំណើរការបានហើយ; ការត្រួតពិនិត្យអវិជ្ជមានធ្វើឲ្យច្បាប់ដែនកំណត់ឆ្លងសញ្ញាដំណើរការបានជាក់ស្តែងមិនទុកឲ្យវាជាអក្សរទេ; បន្ទាត់ចុងក្រោយបង្ហាញការបញ្ចូលខូចដែលត្រឡប់ទៅ `False` ប្រៀបធៀបទៅនឹងការបញ្ចេញករណីលើកលែងមិនបានកើតឡើង។  


## ផ្នែកទី 2៖ បំនាស់លើបង្កាន់ដៃ

គោលបំណងសរុបនៃបង្កាន់ដៃគឺថាវា​អាចបង្ហាញបានថា​មានការ​បំនាស់។ យើងចង់បញ្ជាក់វា។

យើង​នឹង​កែប្រែ​តួអក្សរមួយត្រឹមត្រូវ​​នៅលើបង្កាន់ដៃ និងមើល​ការ​ផ្ទៀងផ្ទាត់​ដែល​​បរាជ័យ។


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### តើមានអ្វីកើតឡើង?

នៅពេលយើងបំលែង `policy_id` អ៊ីមែលបញ្ជាក់បានផ្លាស់ប្ដូរ។ ហត្ថលេខា (ដែលមានលើអ៊ីមែលបញ្ជាក់ដើម) មិនត្រូវគ្នាទៀត។ ការត្រួតពិនិត្យត្រឹមត្រូវត្រលប់ពី `False`។

គ្មានវិធីណាដែលអាចបំលែងវាលដែលបានហត្ថលេខា ដែលមានន័យថាអ្វីៗដែលនៅខាងក្រៅវិស័យ `signature` ហើយវានៅតែអាចបញ្ជាក់បានតាមគាកូនសោជាសាធារណៈរបស់អ្នកចេញផ្សាយ ពុំដល់ពេលចោរប្លន់មានគាកូនសោឯកជនរបស់អ្នកចេញផ្សាយ។ វិស័យ `signature` មិនត្រូវបានហត្ថលេខាទេ៖ `sig` គឺជាហត្ថលេខា ហើយ `verify_receipt` មិនគិត `public_key` ដូច្នេះការកែប្រែវាលនោះតែក៏មិនបម្លែងអ្វីនោះទេ។ ស្របពេលដែលគាកូនសោឯកជននៅក្នុងKey Vault ហើយអ្នកបញ្ជាក់ប្រើគាដូចជាដែលចេញផ្សាយដោយឯករាជ្យនៃអ្នកចេញផ្សាយ ការផ្លាស់ប្ដូរវាលដែលបានហត្ថលេខា គឺមិនអាចលាក់បានទេ។ លក្ខខ័ណ្ឌនោះមានសារៈសំខាន់៖ ក្រឡាចត្រង្គបន្ទាប់បង្ហាញអំពីវិក័យប័ត្រដែលបានកែសម្រួលដែលបញ្ជាក់បានយ៉ាងល្អប្រសើរតាមគាផ្សេង។

សូមសាកល្បងដោយខ្លួនឯង៖ កែប្រែ `tool_name` ឬ `agent_id` ឬ `timestamp` ក្នុងក្រឡាចត្រង្គខាងលើ ហើយរត់ម្តងទៀត។ ការផ្លាស់ប្ដូរនៅវាលដែលបានហត្ថលេខា រាល់ការផ្លាស់ប្ដូរទាំងអស់នាំឲ្យវិក័យប័ត្រមានអត្តសញ្ញាណមិនត្រឹមត្រូវ។


### ការបំបែកដែលហត្ថលេខាមិនចាប់បាន

អ្នកវាយប្រហារខាងលើមិនបានធ្វើការចុះហត្ថលេខាថ្មីឡើយ។ អ្នកដែលប្រុងប្រយ័ត្នជាងនឹង៖ កែប្រែវិក្កយបត្រ,
ចុះហត្ថលេខាមួយជាមួយកូនសោដែលពួកគេទើបបង្កើតបានហើយដាក់កូនសោនោះនៅក្នុង
`public_key` ឆាក។ លទ្ធផលគឺជាវិក្កយបត្រមួយដែលហត្ថលេខាដែលមានគុណភាពពិតប្រាកដសម្រាប់មាតិការបស់វា។


ប្រតិបត្តិនេះ ហើយមើលថា តើការត្រួតពិនិត្យពីរបណ្តាងណាមួយចាប់បានវា។


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


អ្នកគួរតែឃើញ `True` បន្ទាប់មក `False`។ បន្ទាត់ទាំងបី `assert` ធ្វើឱ្យវា
ជាសាកល្បង៖ សែលនេះរំលោះកំហុស ប្រសិនបើការបន្លំដែលបានបង្កើតឡើងពិតជាត្រូវគ្នានឹង
Key របស់អ្នកផ្តល់សមត្ថកិច្ច។

ការបន្លំនេះមានភាពមានតុល្យភាពផ្ទៃក្នុង។ ស្ដាពាក្យរបស់វាពិតជាត្រូវគ្នាទៅនឹង
ខ្លឹមសារនិង key ដែលវាអ្នកកាន់, ដូច្នេះអ្នកពិនិត្យដែលអាន key ចេញពី
សំបុត្រត្រឡប់វាថាជាផ្លូវការ។ អ្វីមួយទាក់ទងនឹងសំបុត្រមិនមែនមើលឃើញខុសទេ ព្រោះ
អ្វីៗទាំងអស់នៅលើវាមិនមែនមិនមានការស្របគ្នាឡើយ។

វត្ថុប៉ុន្មានតែលក់វាជាប់គ្នាគឺការត្រួតពិនិត្យប្រឆាំងនឹង key ដែលឆ្នោតគ្មានវិធី
គាប់បញ្ចូល។ នេះហើយហេតុផលដែល `verify_receipt` ទទួល key ជាអាគុយម៉ង់៖ មុខងារ
ដែលមានហត្ថលេខា `verify_receipt(receipt)` មិនអាចសរសេរបានយ៉ាងទំនុកចិត្តទេ,
ព្រោះ key ផ唯一នោះគឺ key ដែលឆ្នោតបានផ្គត់ផ្គង់។

សំបុត្រដូចគ្នាត្រូវបានរក្សាទុកជាថ្មីជា `sample_receipts/04_resigned_receipt.json` ប្រសិនបើអ្នក
ចង់ពិចារណាវាពីក្រៅសៀវភៅកំណត់ត្រា។


## ផ្នែកទី 3: បង្កប់បង្កាន់ដៃជាសរសេរ

បង្កាន់ដៃមួយសម្រាប់ការប្រតិបត្តិការមួយ។ភ្នាក់ងារច្រើននាក់អាចធ្វើការប្រតិបត្តិការជាច្រើន។ ដើម្បីធានាថាជួរប្រតិបត្តិការទាំងមូលគ្មានការប្តូរផ្លាស់ប្តូរ យើងភ្ជាប់បង្កាន់ដៃនីមួយៗជាមួយបង្កាន់ដៃមុនដោយបញ្ចូល សំណុំលេខកូដរបស់បង្កាន់ដៃមុននៅក្នុងទិន្នន័យរបស់បង្កាន់ដៃថ្មី។

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

ប្រសិនបើមាននរណាដក ឬប្ដូរតំរូវបង្កាន់ដៃនៅក្នុងជួរដែលបានបង្ហាញ ភ្ជាប់បន្ទាប់នឹងខូចខាត។ ការរកឃើញការកាត់បន្ថយឬក៏ការសរសេរឡើងវិញដោយអ្នកចុះហត្ថលេខាក៏ត្រូវការបញ្ចូលក្បាលបច្ចុប្បន្នដែលបានចាក់សោរដើម្បីការពារ ផcheckpointខាងក្រៅឬការប្តេជ្ញាចិត្តអ្នកប្រើប្រាស់ដែលបានផ្ទៀងផ្ទាត់។ 


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


បន្ទាប់មកបំផ្លាញខ្សែសង្វាក់ដោយការបំរាស់បំរួលវិក្កយប័ត្រកណ្តាលហើយធ្វើការត្រួតពិនិត្យឡើងវិញ។ វិក្កយប័ត្រដែលបានបំរាស់បំរួលត្រូវបរាជ័យការត្រួតពិនិត្យហត្ថលេខារបស់វា ហើយវិក្កយប័ត្របន្ទាប់ត្រូវបរាជ័យការត្រួតពិនិត្យខ្សែសង្វាក់របស់វា (ព្រោះ `previous_receipt_hash` របស់វាមិនត្រូវគ្នា នឹង hash របស់វិក្កយប័ត្រកណ្តាលដែលបានកែប្រែទៀតទេ)។


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


សៀវភៅបង្កាន់ដៃ 0 នៅតែបញ្ជាក់បាន (វាមិនត្រូវបានកែប្រែ ហើយមិនមានអ្នកជំរុញមុនសម្រាប់ពឹងផ្អែកលើទេ)។ សៀវភៅបង្កាន់ដៃ 1 មិនអាចផ្ទៀងផ្ទាត់ហត្ថលេខារបស់វាបានដោយសារយើងបានប្ដូរ `tool_args_hash`។ សៀវភៅបង្កាន់ដៃ 2 មិនអាចផ្ទៀងផ្ទាត់តំណភ្ជាប់ប្រព័ន្ធដោយសារថា `previous_receipt_hash` របស់វាត្រូវបានគណនារួចរួមទៅនឹងសៀវភៅបង្កាន់ដៃ 1 ដើម (ដែលឥឡូវនេះបានកែប្រែ)។

ទោះបីជាអ្នកវាយប្រហារបានធ្វើការហត្ថលេខាឡើងវិញលើសៀវភៅបង្កាន់ដៃ 1 ដែលបានកែប្រែ ក៏ហត្ថលេខាថ្មីនោះមិនអាចផ្ទៀងផ្ទាត់បានជាមួយកូនសោរបស់អ្នកចេញផ្សាយទេ លុះត្រাঁពួកគេមិនកាន់កូនសោរឯកជនរបស់អ្នកចេញផ្សាយ (ការបញ្ចេញហត្ថលេខាឡើងវិញជាមួយគ្រាប់សោររបស់ខ្លួនពួកគេវាយតម្លៃថាបរាជ័យ ដូចបានបង្ហាញនៅផ្នែក 2) ហើយការមិនត្រូវគ្នានៃតំណភ្ជាប់ប្រព័ន្ធក្នុងសៀវភៅបង្កាន់ដៃ 2 នឹងនៅតែបង្ហាញពីការផ្លាស់ប្តូរ។ ដើម្បីលាក់ការផ្លាស់ប្តូរ អ្នកវាយប្រហារត្រូវតែធ្វើការហត្ថលេខាឡើងវិញលើសៀវភៅបង្កាន់ដៃគ្រប់ទីពីនឹងចាប់ផ្តើមពីចំណុចកែប្រែ ដែលតម្រូវឱ្យមានការកាន់កាប់ជាប់កូនសោឯកជនរបស់អ្នកចេញផ្សាយ។


## ផ្នែក ៤៖ លុបឧបករណ៍ភ្នាក់ងារមួយជាមួយនឹងការចុះហត្ថលេខារបង្កាន់ដៃ

ក្នុងការដាក់បញ្ចូលជាក់ស្តែង អ្នកមិនចង់ឱ្យអ្នកនិពន្ធភ្នាក់ងារផ្ទាល់ខ្លួនរាល់គ្នានៅតែចងចាំហៅ `make_receipt` ទេ។ អ្នកចង់ឱ្យការចុះហត្ថលេខារបង្កាន់ដៃកើតឡើងដោយស្វ័យប្រវត្តិសម្រាប់ការហៅឧបករណ៍រាល់មួយ។

នេះគឺជារចនាប័ទ្មងាយស្រួលបំផុត៖ ជាក្លាសថ្នាក់រុំមួយដែលទទួលយកមុខងារឧបករណ៍ណាមួយដែលអាចហៅបាន ហើយបម្លែងវាទៅជាម៉ូដែលដែលបញ្ចេញរបង្កាន់ដៃ។ វាគ្របដណ្តប់ទៅលើស៊ុមភ្នាក់ងារណាមួយស្របតាម Microsoft Agent Framework (`agent_framework.foundry`)។

បើអ្នកមិនមានគម្រោង Microsoft Foundry តែម្តង ការបង្ហាញន៍មោឃៈក្នុងតំបន់ខាងក្រោមនៅតែបង្ហាញរចនាប័ទ្មនេះបាន។


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### ការរួមបញ្ចូលជាមួយស៊ុមភ្នាក់ងាររបស់ Microsoft

កម្មវិធីខ្សែបន្ទាត់ `ReceiptedTool` ខាងលើនេះគឺមិនស្ថិតក្នុងស៊ុមណាមួយទេ។ ដើម្បីប្រើវាក្នុងភ្នាក់ងារដែលបានបង្កើតដោយស៊ុមភ្នាក់ងាររបស់ Microsoft, ចុះបញ្ជីមុខងារwrapped ជាឧបករណ៍មួយ។ គំនូរ (អ្នកនឹងជំនួស mock ជាការចុះបញ្ជីឧបករណ៍ Microsoft Foundry ពិតប្រាកដ):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

ស៊ុមភ្នាក់ងារមិនត្រូវការចេះអ្វីក្រុមហ៊ុន receipt ទេ។ ការចុះហត្ថលេខាលើ receipt ត្រូវបានចាក់ដុតជុំវិញឧបករណ៍ មិនមែនភ្ជាប់ ទៅក្នុងស៊ុមនោះទេ។ វាគឺជាវិធីដែលអ្នកបន្ថែមប្រភពទៅកូដភ្នាក់ងារដែលមានរួចហើយ ដោយមិនចាំបាច់សរសេរប្ដូរភ្នាក់ងារឡើយ។


## សេចក្ដីសង្ខេប និង thách thứcពង្រីក

អ្នកបាន:

- បង្កើតគូសោ Ed25519 មួយ។
- បានកសាង និងចុះហត្ថលេខាលើបង្កាន់ដៃសម្រាប់ការហៅឧបករណ៍ភ្នាក់ងារ។
- បានផ្ទៀងផ្ទាត់បង្កាន់ដៃនៅក្រៅបណ្តាញដោយប្រើសោសាធារណៈរបស់អ្នកចេញផ្សាយ ដែលបានទទួលដោយឯករាជ្យពីបង្កាន់ដៃ។
- បានកែប្រែក្នុងបង្កាន់ដៃ និងបានសង្កេតឃើញការផ្ទៀងផ្ទាត់បរាជ័យ។
- បានកសាងជួរលំនាំ hash-chained របស់បង្កាន់ដៃបី។
- បានកែប្រែកណ្ដាលខ្សែសង្វាក់ ហើយបានសង្កេតឃើញទាំងការបរាជ័យហត្ថលេខា និងការបរាជ័យខ្សែសង្វាក់។
- បញ្ចូលមុខងារឧបករណ៍ជាមួយការចុះហត្ថលេខាអគ្គិសនីដោយស្វ័យប្រវត្តិបង្កាន់ដៃ។

**ការប្រកួតបន្ថែម។** ពង្រីកស្កីមារបង្កាន់ដៃជាមួយ `request_id` ដើមទូលាយ (UUID សម្រាប់ការតាមដានចែកចាយ)។ បន្ទាប់មកកែប្រែ `make_receipt` ដើម្បីបញ្ចូលវា ហើយបញ្ជាក់ថាការបង្កាន់ដៃនៅតែត្រូវបានផ្ទៀងផ្ទាត់ពីដល់ចុងបញ្ចប់។ បន្ទាប់មកកែប្រែដើមទូលាយបន្ទាប់ពីចុះហត្ថលេខា ហើយបញ្ជាក់ថាការផ្ទៀងផ្ទាត់បរាជ័យ។ នេះធ្វើឲ្យអ្នកត្រូវយល់ខ្លួនថាតើបៃណ៍នីមួយៗនៃការប៉ះពាល់នូវអក្សរ​ត្រឹមត្រូវរបស់ការវាយបញ្ចូលសញ្ញាដែលមានឥទ្ឋិពលដល់ហត្ថលេខា។

**ព្រំដែនសំខាន់។** នៅក្រោម context key និងខ្សែសង្វាក់ដែលត្រូវបានចាក់សោរ receipts បញ្ជាក់ពីការចុះហត្ថលេខា ការអាស្រ័យលើខ្លឹមសារ និងតំណភ្ជាប់ទៅកាន់អ្នកកន្សោមមុន។ វាមិនបញ្ជាក់ពីការអនុវត្ត ការពិតប្រភព ពេលវេលាទុកចិត្ត ភាពពេញលេញ ការវាយតម្លៃគោលការណ៍ ឬភាពត្រឹមត្រូវទេ។ Receipts ជាមូលដ្ឋានមួយ។ អាជ្ញាធរជាប្រព័ន្ធដែលអ្នកបង្កើតលើវា។

អាន README មុខវិជ្ជារឡើងវិញដោយយកព្រំដែននោះមកគិត។ កំហុសសំខាន់ដែលក្រុមភាគច្រើនធ្វើជាមួយ receipts គឺការសន្និដ្ឋានថា "យើងមាន receipts" ស្មើ "យើងមានអាជ្ញាធរ"។ វាមិនមែនប៉ុន្មានទេ។ Receipts ធ្វើឱ្យអនុវត្តន៍ភាគីអាចត្រួតពិនិត្យបាន។ វាមិនធ្វើឱ្យវាត្រឹមត្រូវឡើយ។


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
